# Object detector evaluation

Measures the pinned COCO detectors (ADR 0014) on nuScenes CAM_FRONT. The headline is recall for
the road users that matter most, pedestrian and cyclist, at the score threshold the system
actually runs (0.5). F1 and AP50 follow, for comparison with the source repo. Logic lives in
`eval/detector.py`, `eval/provenance.py` and `tools/eval_detector.py`, which are tested.

**Data rule:** run outputs can contain RDD2022 and nuScenes images. Clear all outputs before
committing (Kernel, Restart and Clear Outputs). A test fails if a committed notebook has
outputs, machine-specific paths or function definitions.

## Settings

Needs the nuScenes devkit and the weights from `python -m tools.fetch_detector`.
`THREAD_LADDER` times the CPU at each thread count to imitate smaller hosts;
`TIMING_FRAMES = 0` skips the speed test.

In [ ]:
import os
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
NUSCENES_ROOT = REPO / "data/nuscenes/raw"
REPORT_DIR = Path(os.environ.get("EVAL_REPORT_DIR", REPO / "data/nuscenes/derived"))
REPORT = REPORT_DIR / "detector_report.json"
# None = every pinned detector, or a list such as ["fasterrcnn_mobilenet_v3_large_fpn"]
MODELS = None
TIMING_FRAMES = 100
THREAD_LADDER = [os.cpu_count() or 4, 4, 2]
SHIPPED_THRESHOLD = 0.5  # perception.score_threshold, provisional (ADR 0014)
PRIORITY_CLASSES = ("pedestrian", "cyclist")

## Run

Writes the JSON report (numbers and hashes only, no images).

In [ ]:
import contextlib
import io
import json

from tools.eval_detector import main

argv = [
    "--nuscenes-root",
    str(NUSCENES_ROOT),
    "--timing-frames",
    str(TIMING_FRAMES),
    "--thread-ladder",
    *map(str, THREAD_LADDER),
    "--out",
    str(REPORT),
]
if MODELS:
    argv += ["--models", *MODELS]
with contextlib.redirect_stdout(io.StringIO()):
    main(argv)
report = json.loads(REPORT.read_text())
print(f"{report['frames']} frames; held-out scenes: {len(report['held_out_scenes'])}")

## Provenance

What these numbers were measured on; reports with different hashes are not comparable.

In [ ]:
prov = report["provenance"]
print("git commit:", prov["git_commit"], "(uncommitted changes)" if prov["git_dirty"] else "")
print("date (UTC):", prov["date_utc"])
for name, digest in prov["files"].items():
    print(f"  {name:<40} {digest[:16]}")

## Headline: recall for pedestrians and cyclists at the shipped threshold

Evaluable ground truth (at least 40% visible and 32 x 32 px; this tool's own cut-off) on all
frames, at score threshold 0.5. Ranges are 90% Wilson intervals on recall. Few cyclists exist in
nuScenes mini, so expect wide ranges.

In [ ]:
from eval.classifier import wilson_interval

print(f"{'model':<36}{'class':<12}{'recall':>8}{'range (90%)':>14}{'precision':>11}{'objects':>9}")
for name, m in report["models"].items():
    for label in PRIORITY_CLASSES:
        sweep = m["sweep_by_class_evaluable"][label]
        row = next(r for r in sweep if r["threshold"] == SHIPPED_THRESHOLD)
        total = row["n_ground_truth"]
        found = round(row["recall"] * total)
        low, high = wilson_interval(found, total)
        print(
            f"{name:<36}{label:<12}{row['recall']:>8.2f}{f'{low:.2f}-{high:.2f}':>14}"
            f"{row['precision']:>11.2f}{total:>9}"
        )

## All six fusion classes at 0.5

Recall and precision per class, evaluable ground truth.

In [ ]:
for name, m in report["models"].items():
    print(name)
    for label, sweep in m["sweep_by_class_evaluable"].items():
        row = next(r for r in sweep if r["threshold"] == SHIPPED_THRESHOLD)
        print(
            f"  {label:<12} recall {row['recall']:.2f}  precision {row['precision']:.2f}"
            f"  ({row['n_ground_truth']} objects)"
        )

## F1 and AP50

The shipped threshold (0.5) beside the F1-best threshold picked on the training scenes. "Held"
is scored on scenes the threshold never saw (few frames, so a few points are noise).

In [ ]:
print(f"{'model':<36}{'point':<10}{'F1':>7}{'recall':>8}{'precision':>11}")
for name, m in report["models"].items():
    shipped = next(r for r in m["sweep_evaluable"] if r["threshold"] == SHIPPED_THRESHOLD)
    print(
        f"{name:<36}{'0.5':<10}{shipped['f1']:>7.2f}"
        f"{shipped['recall']:>8.2f}{shipped['precision']:>11.2f}"
    )
    best = m["all_frames_evaluable"]
    print(
        f"{'':<36}{m['threshold']:<10.2f}{best['f1']:>7.2f}"
        f"{best['recall']:>8.2f}{best['precision']:>11.2f}"
    )
    held = m["held_out_evaluable"]
    print(f"{'':<36}held: F1 {held['f1']:.2f}, AP50 {held['ap50']:.2f}")

## Day and night

F1 at the F1-best threshold, evaluable ground truth.

In [ ]:
for name, m in report["models"].items():
    day, night = m["day_evaluable"]["f1"], m["night_evaluable"]["f1"]
    print(f"{name:<36} day F1 {day:.2f}  night F1 {night:.2f}")

## Recall by object size

All projected ground-truth boxes, at the F1-best threshold.

In [ ]:
for name, m in report["models"].items():
    print(name)
    for row in m["recall_by_size_all_gt"]:
        high = "up" if row["high"] is None else f"{row['high']:.0f}"
        span = f"{row['low']:.0f} to {high} px^2"
        print(f"  area {span}: recall {row['recall']:.2f} over {row['n']} objects")

## Threshold sweep

Precision, recall and F1 at each score threshold (evaluable ground truth, all frames).

In [ ]:
import matplotlib.pyplot as plt

count = len(report["models"])
fig, axes = plt.subplots(1, count, figsize=(6 * count, 4), squeeze=False)
for ax, (name, m) in zip(axes.ravel(), report["models"].items(), strict=False):
    sweep = m["sweep_evaluable"]
    for key in ("precision", "recall", "f1"):
        ax.plot([r["threshold"] for r in sweep], [r[key] for r in sweep], label=key)
    ax.axvline(SHIPPED_THRESHOLD, color="black", linestyle="-", label="shipped")
    ax.axvline(m["threshold"], color="grey", linestyle=":", label="F1-best (train scenes)")
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("score threshold")
    ax.legend()
plt.show()

## CPU speed: thread ladder

One frame at a time after warm-up. PASS means a p95 of at most 2 s per image (ADR 0014).
The real host is not known (D5), so each rung imitates a host with that many threads.

In [ ]:
for name, m in report["models"].items():
    print(name)
    for rung in m.get("cpu_latency_ladder", []):
        verdict = "PASS" if rung["passed"] else "FAIL"
        print(
            f"  {rung['threads']:>3} threads: p50 {rung['p50_s']:.2f} s, "
            f"p95 {rung['p95_s']:.2f} s -> {verdict}"
        )
    if "cpu_latency_ladder" not in m:
        print("  speed not measured")